# Reddit Live Sentiment — Streaming Ingestion

The **real-time** half of the project. Where the batch pipeline analyses a fixed YouTube
corpus, this notebook ingests an *unbounded* stream of new Reddit comments, stores them in
MongoDB, and visualises sentiment as it evolves minute by minute.

**Architecture:** Reddit (PRAW) → MongoDB buffer → Jupyter processing → live Matplotlib
plots. MongoDB acts as a simple, schema-flexible buffer so the stream keeps running even
if the analysis kernel restarts.

**Note on running:** requires a running local MongoDB instance and Reddit API credentials
(the placeholders in the streaming cell below).

## 1. Connect to MongoDB and reset the collection

Connect to a local MongoDB instance and clear any previous comments so each run starts
from a clean buffer.

In [ ]:
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["reddit_sentiment"]
col = db["reddit_comments"]

# Clear old data so each run is fresh
result = col.delete_many({})
print(f"Cleared {result.deleted_count} existing comments from MongoDB.")

## 2. Start the Reddit stream

Connect to Reddit via PRAW and open a comment stream across the target subreddits
(three genre communities plus two artist-focused ones). A background daemon thread
continuously writes each incoming comment — text, subreddit, and a UTC timestamp — into
MongoDB. Running it as a daemon thread means it stops cleanly when the kernel does.

*Credentials are placeholders; supply your own Reddit API client ID and secret to run.*

In [ ]:
import praw
from pymongo import MongoClient
from datetime import datetime
import threading
import nltk

from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Make sure VADER is available
nltk.download("vader_lexicon")

# MongoDB connection (same as Cell 1)
client = MongoClient("mongodb://localhost:27017/")
col = client["reddit_sentiment"]["reddit_comments"]

# Reddit API (your existing credentials)
reddit = praw.Reddit(
    client_id="YOUR_CLIENT_ID_HERE",
    client_secret="YOUR_CLIENT_SECRET_HERE",
    user_agent="LiveStream/1.0"
)

# Subreddits to monitor
SUBREDDITS = [
    "hiphopheads", 
    "indieheads", 
    "popheads",
    "drizzy",            # Drake-focused subreddit
    "kendricklamar"      # Kendrick-focused subreddit
]

SUBREDDIT_STR = "+".join(SUBREDDITS)

def stream_reddit():
    """
    Background thread:
    continuously listens to new comments on the selected subreddits
    and writes them into MongoDB.
    """
    print("Streaming from:", SUBREDDIT_STR)
    for c in reddit.subreddit(SUBREDDIT_STR).stream.comments(skip_existing=True):
        col.insert_one({
            "text": c.body,
            "subreddit": str(c.subreddit).lower(),
            "published_at": datetime.utcnow().isoformat()
        })

# Start streaming in a daemon thread so it stops with the kernel
threading.Thread(target=stream_reddit, daemon=True).start()
print("Streamer thread started. Wait a bit for comments to accumulate.")

## 3. Architecture reference

A plain-text summary of how the streaming pipeline fits together and how it mirrors the
batch pipeline.

In [ ]:
architecture_description = """
REAL-TIME STREAMING ARCHITECTURE (Reddit → MongoDB → Jupyter)

1. Reddit API (PRAW)
   - Connects to r/hiphopheads, r/indieheads, r/popheads.
   - Uses stream.comments() to receive new comments as they are posted.

2. Ingestion Layer (MongoDB)
   - Each comment is stored as a document:
     { text, subreddit, published_at }.
   - This acts as a simple message buffer / data lake.

3. Processing & Analytics (Jupyter Notebook)
   - Every few seconds, the notebook:
     - Reads all comments from MongoDB.
     - Cleans the text.
     - Applies VADER and TextBlob sentiment analysis.
     - Computes rolling averages per subreddit.

4. Visualisation Layer (Matplotlib)
   - Three panels, one per subreddit.
   - Live plots of rolling VADER and TextBlob sentiment.
   - Overlaid statistics: number of comments, positive/neutral/negative counts.
   - This makes it easy to compare real-time mood between genres.

This mirrors the batch pipeline (YouTube → JSONL → Spark → Sentiment),
but here the data is unbounded and arrives continuously.
"""
print(architecture_description)

## 4. Processing helpers

Text cleaning and the scoring functions used by the live loop: VADER compound, TextBlob
polarity and subjectivity, and a numeric→label mapper. `ROLL` sets the rolling-average
window (in comments) used to smooth the noisy live signal.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time
from textblob import TextBlob

from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Rolling window size in NUMBER OF COMMENTS
ROLL = 20
MAX_POINTS = 400  # only plot most recent N comments per subreddit


def clean_text(t: str) -> str:
    """Simple cleaning: lowercase, remove URLs, non-letters, collapse spaces."""
    if t is None:
        return ""
    t = t.lower()
    t = re.sub(r"http\S+", "", t)          # remove URLs
    t = re.sub(r"[^a-zA-Z\s]", " ", t)     # keep only letters/spaces
    t = re.sub(r"\s+", " ", t).strip()     # collapse whitespace
    return t


def vader_score(text: str) -> float:
    """Return VADER compound score in [-1, 1]."""
    return float(sia.polarity_scores(text)["compound"])


def tb_polarity(text: str) -> float:
    """Return TextBlob polarity score in [-1, 1]."""
    return float(TextBlob(text).sentiment.polarity)


def tb_subjectivity(text: str) -> float:
    """Return TextBlob subjectivity score in [0, 1]."""
    return float(TextBlob(text).sentiment.subjectivity)


def label_sentiment(score: float) -> str:
    """Map numeric sentiment score to positive / neutral / negative."""
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

## 5. The live plot

Pull the latest comments from MongoDB, score them, keep the most recent `MAX_POINTS` per
subreddit, and draw a three-panel figure of rolling VADER and TextBlob sentiment with
per-subreddit counts and mean subjectivity.

In [ ]:
def draw_plot():
    """
    Pulls latest comments from MongoDB, computes sentiment and subjectivity,
    and draws a 3-panel figure (one per subreddit) with rolling averages
    and basic statistics.
    """
    docs = list(col.find().sort("published_at", 1))
    if not docs:
        clear_output(wait=True)
        print("Waiting for data from Reddit...")
        return

    df = pd.DataFrame(docs)

    # Convert time to datetime
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True)

    # Clean text + sentiment + subjectivity
    df["clean_text"] = df["text"].apply(clean_text)
    df["vader"] = df["clean_text"].apply(vader_score)
    df["textblob"] = df["clean_text"].apply(tb_polarity)
    df["subjectivity"] = df["clean_text"].apply(tb_subjectivity)

    # Limit to most recent MAX_POINTS per subreddit for clearer plots
    frames = []
    for sub in SUBREDDITS:
        d_sub = df[df["subreddit"] == sub].sort_values("published_at")
        if len(d_sub) > MAX_POINTS:
            d_sub = d_sub.iloc[-MAX_POINTS:]
        frames.append(d_sub)
    df = pd.concat(frames) if frames else df

    # Plotting
    clear_output(wait=True)
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
    fig.suptitle("Real-Time Sentiment by Subreddit (VADER + TextBlob)", fontsize=14)

    for ax, sub in zip(axes, SUBREDDITS):
        d = df[df["subreddit"] == sub]

        if d.empty:
            ax.set_title(f"{sub} (no data yet)")
            ax.set_ylim(-1, 1)
            ax.grid(True, alpha=0.3)
            continue

        # Rolling averages
        v_avg = d["vader"].rolling(ROLL, min_periods=1).mean()
        t_avg = d["textblob"].rolling(ROLL, min_periods=1).mean()

        ax.plot(d["published_at"], v_avg, "r-", label="VADER (rolling)")
        ax.plot(d["published_at"], t_avg, "b-", label="TextBlob (rolling)")

        # Basic stats
        n_comments = len(d)
        pos = (d["vader"] >= 0.05).sum()
        neg = (d["vader"] <= -0.05).sum()
        neu = n_comments - pos - neg
        avg_subj = d["subjectivity"].mean()

        stats_text = f"N={n_comments} | +:{pos}  0:{neu}  -:{neg} | mean subj={avg_subj:.2f}"
        ax.text(0.01, 0.93, stats_text, transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.6))

        ax.set_title(sub)
        ax.set_ylim(-1, 1)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="lower left")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

## 6. Run the live loop

Redraw every five seconds until interrupted. Interrupt the kernel (stop the cell) to halt
the visualisation; the background streamer keeps buffering to MongoDB.

In [ ]:
import time

print("Starting live visualisation loop. Interrupt the cell to stop.")

try:
    while True:
        draw_plot()
        time.sleep(5)   # update every 5 seconds
except KeyboardInterrupt:
    print("Stopped live visualisation loop.")